# Review and soft-merge over-split units in one SpikeInterface session

This workflow is independent of cross-session stitching. `SpikeInterfaceSessionMerger` runs split-half UnitMatch, requires both directional probabilities to exceed the threshold, applies the ISI safety check, collects manual approval, soft-merges approved pairs, and saves the resulting analyzer. If curation produced complementary analyzers, pass both plus the final unit IDs from `metrics.index`; replaced source units absent from the metrics table are excluded.

In [ ]:
from pathlib import Path

import spikeinterface.full as si

from UnitMatchPy.spikeinterface_merging import SpikeInterfaceSessionMerger

USE_SYNTHETIC_DATA = True
ANALYZER_PATH = Path("path/to/sorting_analyzer")
MERGED_ANALYZER_PATH = Path("merged_sorting_analyzer")
MATCH_THRESHOLD = 0.5
CENSORED_PERIOD_MS = 0.5

In [ ]:
def make_synthetic_analyzer(seed=410):
    recording, sorting = si.generate_ground_truth_recording(
        durations=[5.0], sampling_frequency=30_000.0,
        num_channels=16, num_units=8, seed=seed,
    )
    analyzer = si.create_sorting_analyzer(
        sorting=sorting, recording=recording, format="memory", sparse=True
    )
    analyzer.compute(
        "random_spikes", method="uniform", max_spikes_per_unit=200, seed=seed
    )
    analyzer.compute("waveforms", ms_before=1.0, ms_after=1.0)
    return analyzer


analyzer = (
    make_synthetic_analyzer()
    if USE_SYNTHETIC_DATA
    else si.load_sorting_analyzer(ANALYZER_PATH)
)
# For complementary original/replacement analyzers, use:
# analyzers = [original_analyzer, merged_splitted_analyzer]
# merger = SpikeInterfaceSessionMerger(analyzers, unit_ids=metrics.index)
# The single-analyzer example below remains equivalent.
merger = SpikeInterfaceSessionMerger(
    analyzer,
    match_threshold=MATCH_THRESHOLD,
    censored_period_ms=CENSORED_PERIOD_MS,
)
merge_groups = merger.compute_proposals()
print(f"Proposed merge groups: {merge_groups}")

## Manual review

Approve or reject every proposal. The next step refuses to continue while any group remains undecided.

In [ ]:
review_widget = merger.display_review()

## Apply and save

Run this cell after finishing the review. `binary_folder` persists the soft-merged sorting without hard recomputation. Single-analyzer curation propagates existing extensions; a composite analyzer retains the final sorting and recording metadata but cannot combine analyzer extensions that originated in separate stores.

In [ ]:
merged_analyzer = merger.apply_merges()
saved_analyzer = merger.save(
    MERGED_ANALYZER_PATH, format="binary_folder", overwrite=True
)
print(f"Approved groups: {merger.approved_groups}")
print(f"Units before: {len(analyzer.unit_ids)}")
print(f"Units after:  {len(saved_analyzer.unit_ids)}")
print(f"Saved to: {MERGED_ANALYZER_PATH.resolve()}")